In [1]:

import numpy as np
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/celeb-df-v2/List_of_testing_videos.txt
/kaggle/input/celeb-df-v2/YouTube-real/00238.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00152.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00269.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00209.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00297.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00096.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00255.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00230.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00259.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00012.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00219.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00003.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00147.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00241.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00260.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00128.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00243.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00248.mp4
/kaggle/input/celeb-df-v2/YouTube-real/00207.mp4
/kaggle/input/ce

In [2]:
import os
import cv2
import random
import shutil
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import glob

In [3]:
pip install torch torchvision timm numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
dataset_path = '/kaggle/input/celeb-df-v2'
real_videos_path = os.path.join(dataset_path, 'Celeb-real')
fake_videos_path = os.path.join(dataset_path, 'Celeb-synthesis')

print(f"real samples: {len(os.listdir(os.path.join(dataset_path, real_videos_path)))}")
print(f"fake samples: {len(os.listdir(os.path.join(dataset_path, fake_videos_path)))}")

real samples: 590
fake samples: 5639


In [5]:
SPLIT_OUTPUT_DIR = "/kaggle/working/split_videos"

REAL_TRAIN_DIR = os.path.join(SPLIT_OUTPUT_DIR, "REAL/train")
REAL_VAL_DIR = os.path.join(SPLIT_OUTPUT_DIR, "REAL/val")
REAL_TEST_DIR = os.path.join(SPLIT_OUTPUT_DIR, "REAL/test")

FAKE_TRAIN_DIR = os.path.join(SPLIT_OUTPUT_DIR, "Fake/train")
FAKE_VAL_DIR = os.path.join(SPLIT_OUTPUT_DIR, "Fake/val")
FAKE_TEST_DIR = os.path.join(SPLIT_OUTPUT_DIR, "Fake/test")

def split_and_balance_videos(source_dir, train_dir, val_dir, test_dir, train_ratio=0.7, val_ratio=0.1, balance_ratio=None):
    
    # Read videos from the source directory
    videos = [os.path.join(source_dir, video) for video in os.listdir(source_dir) if video.endswith(('.mp4', '.avi', '.mov'))]
    
    if not videos:
        print(f"No videos found in {source_dir}")
        return

    # Apply balance if balance_ratio is set
    if balance_ratio is not None:
        videos = train_test_split(videos, test_size=balance_ratio, random_state=42)[1]

    # Split videos into train and remaining sets
    train_videos, remaining_videos = train_test_split(videos, test_size=(1 - train_ratio), random_state=42)
    
    # Split remaining videos into validation and test sets
    val_ratio_adjusted = val_ratio / (1 - train_ratio)  # Adjusted validation ratio
    val_videos, test_videos = train_test_split(remaining_videos, test_size=(1 - val_ratio_adjusted), random_state=42)

    # Copy videos to respective directories
    for video_set, output_dir in zip([train_videos, val_videos, test_videos], [train_dir, val_dir, test_dir]):
        os.makedirs(output_dir, exist_ok=True)
        for video_path in video_set:
            shutil.copy(video_path, output_dir)

    print(f"Splitting completed for {source_dir}:")
    print(f"  Train: {len(train_videos)} videos -> {train_dir}")
    print(f"  Validation: {len(val_videos)} videos -> {val_dir}")
    print(f"  Test: {len(test_videos)} videos -> {test_dir}")


In [6]:
split_and_balance_videos(real_videos_path, REAL_TRAIN_DIR, REAL_VAL_DIR, REAL_TEST_DIR)

split_and_balance_videos(fake_videos_path, FAKE_TRAIN_DIR, FAKE_VAL_DIR, FAKE_TEST_DIR, balance_ratio=0.11)


Splitting completed for /kaggle/input/celeb-df-v2/Celeb-real:
  Train: 412 videos -> /kaggle/working/split_videos/REAL/train
  Validation: 59 videos -> /kaggle/working/split_videos/REAL/val
  Test: 119 videos -> /kaggle/working/split_videos/REAL/test
Splitting completed for /kaggle/input/celeb-df-v2/Celeb-synthesis:
  Train: 434 videos -> /kaggle/working/split_videos/Fake/train
  Validation: 62 videos -> /kaggle/working/split_videos/Fake/val
  Test: 125 videos -> /kaggle/working/split_videos/Fake/test


In [7]:
import os
import cv2

def extract_frames(source_dir, output_dir, frame_rate=30):
    
    print(f"Processing videos in: {source_dir}")
    os.makedirs(output_dir, exist_ok=True)

    video_files = [f for f in os.listdir(source_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
    if not video_files:
        print(f"No video files found in {source_dir}.")
        return

    for video_file in video_files:
        video_path = os.path.join(source_dir, video_file)

        # Create a subdirectory for the video's frames
        video_frames_dir = os.path.join(output_dir, os.path.splitext(video_file)[0])
        os.makedirs(video_frames_dir, exist_ok=True)

        # Extract frames from the video
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % frame_rate == 0:
                frame_path = os.path.join(video_frames_dir, f"frame_{frame_count}.jpg")
                cv2.imwrite(frame_path, frame)  # Save frame as an image
                frame_count += 1
        cap.release()
        print(f"Frames extracted for video: {video_file}")
    print(f"Frame extraction completed for: {source_dir}")

# Correct paths for validation directories
REAL_VAL_DIR = "/kaggle/working/split_videos/REAL/val"
FAKE_VAL_DIR = "/kaggle/working/split_videos/Fake/val"

# Output directory for extracted frames
OUTPUT_FRAMES_DIR = "/kaggle/working/extracted_frames/val"

# Extract frames for REAL and Fake validation videos
extract_frames(REAL_VAL_DIR, os.path.join(OUTPUT_FRAMES_DIR, "REAL"))
extract_frames(FAKE_VAL_DIR, os.path.join(OUTPUT_FRAMES_DIR, "Fake"))


Processing videos in: /kaggle/working/split_videos/REAL/val
Frames extracted for video: id19_0005.mp4
Frames extracted for video: id3_0007.mp4
Frames extracted for video: id7_0007.mp4
Frames extracted for video: id37_0007.mp4
Frames extracted for video: id61_0001.mp4
Frames extracted for video: id57_0000.mp4
Frames extracted for video: id53_0003.mp4
Frames extracted for video: id7_0003.mp4
Frames extracted for video: id22_0003.mp4
Frames extracted for video: id30_0006.mp4
Frames extracted for video: id56_0004.mp4
Frames extracted for video: id37_0005.mp4
Frames extracted for video: id50_0002.mp4
Frames extracted for video: id47_0009.mp4
Frames extracted for video: id45_0008.mp4
Frames extracted for video: id58_0007.mp4
Frames extracted for video: id11_0008.mp4
Frames extracted for video: id27_0003.mp4
Frames extracted for video: id13_0006.mp4
Frames extracted for video: id17_0004.mp4
Frames extracted for video: id8_0002.mp4
Frames extracted for video: id3_0008.mp4
Frames extracted for 

In [8]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

# Define transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),  # Adjust the size if needed
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet mean
                         std=[0.229, 0.224, 0.225])    # ImageNet std
])

Using device: cuda


In [9]:
# Custom Dataset Class
class CustomDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image_path = self.file_paths[idx]
        image = cv2.imread(image_path)
    
        # Check if the image was loaded successfully
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

In [10]:
# Function to evaluate the model
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.float().to(device)

            outputs = model(inputs).squeeze()
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    # Calculate metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}")
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1}


In [11]:
# Function for simulated annealing to find the best learning rate
def simulated_annealing_lr(model, criterion, dataloader, upper_lr=0.001, epochs=5):
    best_lr = upper_lr
    best_loss = float('inf')
    current_lr = random.uniform(1e-5, upper_lr)

    for epoch in range(epochs):
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=current_lr)
        model.train()
        total_loss = 0

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().to(device)

            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_lr = current_lr

        current_lr *= 0.9  # Decrease learning rate
        print(f"Iteration {epoch+1} ---------- Best LR = {best_lr} ---------- Loss = {best_loss}")

    return best_lr


In [12]:
# Function to create and configure the model
def create_model(model_name, num_classes=1, fine_tune=False, unfreeze_layers=2, add_extra_conv=False):
    if model_name == "vgg16":
        model = models.vgg16(pretrained=True)
        if fine_tune:
       
            for param in model.parameters():
                param.requires_grad = False
           
            for param in model.features[-(unfreeze_layers*2):].parameters():
                param.requires_grad = True
        if add_extra_conv:
          
            model.features.add_module("extra_conv", nn.Conv2d(512, 512, kernel_size=3, padding=1))
            model.features.add_module("extra_relu", nn.ReLU(inplace=True))

        num_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(num_features, num_classes)

    elif model_name == "resnet50":
        model = models.resnet50(pretrained=True)
        if fine_tune:
          
            for param in model.parameters():
                param.requires_grad = False
         
            layers = list(model.children())[:-1]
            for layer in layers[-unfreeze_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
        if add_extra_conv:
           
            model.layer4.add_module("extra_conv", nn.Conv2d(2048, 2048, kernel_size=3, padding=1))
            model.layer4.add_module("extra_relu", nn.ReLU(inplace=True))

        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    elif model_name == "densenet121":
        model = models.densenet121(pretrained=True)
        if fine_tune:
         
            for param in model.parameters():
                param.requires_grad = False
          
            for param in model.features.denseblock4.parameters():
                param.requires_grad = True
        if add_extra_conv:
        
            model.features.add_module("extra_conv", nn.Conv2d(1024, 1024, kernel_size=3, padding=1))
            model.features.add_module("extra_relu", nn.ReLU(inplace=True))

        num_features = model.classifier.in_features
        model.classifier = nn.Linear(num_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=True)
        if fine_tune:
        
            for param in model.parameters():
                param.requires_grad = False
        
            for param in model.features[-(unfreeze_layers):].parameters():
                param.requires_grad = True
        if add_extra_conv:
       
            model.features.add_module("extra_conv", nn.Conv2d(1280, 1280, kernel_size=3, padding=1))
            model.features.add_module("extra_relu", nn.ReLU(inplace=True))

        num_features = model.last_channel
        model.classifier[1] = nn.Linear(num_features, num_classes)

    elif model_name == "inception_v3":
        model = models.inception_v3(pretrained=True, aux_logits=False)
        if fine_tune:
            # Freeze all layers
            for param in model.parameters():
                param.requires_grad = False
            # Unfreeze the last convolutional layers
            for param in model.Mixed_7c.parameters():
                param.requires_grad = True
            if unfreeze_layers > 1:
                for param in model.Mixed_7b.parameters():
                    param.requires_grad = True
        if add_extra_conv:
            # Add an extra convolutional layer
            model.Mixed_7c.add_module("extra_conv", nn.Conv2d(2048, 2048, kernel_size=3, padding=1))
            model.Mixed_7c.add_module("extra_relu", nn.ReLU(inplace=True))

        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    elif model_name == "vision_transformer":
        # Import timm and create vision transformer model
        import timm
        model = timm.create_model('vit_base_patch16_224', pretrained=True)
        if fine_tune:
            # Freeze all layers
            for param in model.parameters():
                param.requires_grad = False
            # Unfreeze the last transformer blocks
            for param in model.blocks[-unfreeze_layers:].parameters():
                param.requires_grad = True
        num_features = model.head.in_features
        model.head = nn.Linear(num_features, num_classes)
        
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    return model


In [ ]:
def k_fold_cross_validation(k, model_name, dataset, batch_size=32, epochs=5, fine_tune=False, unfreeze_layers=2, add_extra_conv=False):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []
    fold = 0

    for train_idx, test_idx in kf.split(dataset):
        fold +=1
        print(f"\nStarting Fold {fold}")

        train_subset = torch.utils.data.Subset(dataset, train_idx)
        test_subset = torch.utils.data.Subset(dataset, test_idx)

        # Create DataLoaders from the subsets
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

        model = create_model(model_name, fine_tune=fine_tune, unfreeze_layers=unfreeze_layers, add_extra_conv=add_extra_conv)
        model = model.to(device)

        criterion = nn.BCEWithLogitsLoss()

        best_lr = simulated_annealing_lr(model, criterion, train_loader)
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=best_lr, momentum=0.9)

        # Training loop
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.float().to(device)  # Convert labels to float

                optimizer.zero_grad()
                outputs = model(images).squeeze()
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

        # Evaluate the model
        metrics = evaluate(model, test_loader)
        fold_metrics.append(metrics)

    # Calculate average metrics
    avg_accuracy = np.mean([m['accuracy'] for m in fold_metrics])
    avg_precision = np.mean([m['precision'] for m in fold_metrics])
    avg_recall = np.mean([m['recall'] for m in fold_metrics])
    avg_f1 = np.mean([m['f1_score'] for m in fold_metrics])
    print(f"\nAverage Metrics for {model_name}:")
    print(f"Accuracy: {avg_accuracy:.4f}")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall: {avg_recall:.4f}")
    print(f"F1 Score: {avg_f1:.4f}")

    # Return the last trained model and metrics
    return model, {'accuracy': avg_accuracy, 'precision': avg_precision, 'recall': avg_recall, 'f1_score': avg_f1}



In [14]:
def run_experiments():
    import traceback  # For detailed error logs
    import pickle     # For saving results to files

  
    TRAIN_DIR = "/kaggle/working/extracted_frames/val"

   
    real_image_paths = glob.glob(os.path.join(TRAIN_DIR, "REAL", "**", "*.jpg"), recursive=True)
    fake_image_paths = glob.glob(os.path.join(TRAIN_DIR, "Fake", "**", "*.jpg"), recursive=True)

    all_files = real_image_paths + fake_image_paths
    all_labels = [0] * len(real_image_paths) + [1] * len(fake_image_paths)  # 0 for REAL, 1 for Fake

   
    dataset = CustomDataset(all_files, all_labels, transform=transform)

    # Models to experiment with
    model_names = ["resnet50", "vgg16", "mobilenet_v2", "densenet121", "inception_v3", "vision_transformer"]
    results = {}
    errors = {}

    for model_name in model_names:
        print(f"\nRunning experiments for model: {model_name}")

        try:
            # Base Model
            print("\n--- Base Model ---")
            base_model, base_metrics = k_fold_cross_validation(
                k=3,
                model_name=model_name,
                dataset=dataset,
                epochs=5,
                fine_tune=False,
                unfreeze_layers=0,
                add_extra_conv=False
            )
            results[f"{model_name}_base"] = base_metrics

            # Save results after each experiment
            with open('results.pkl', 'wb') as f:
                pickle.dump(results, f)

            # Fine-tuned Model
            print("\n--- Fine-tuned Model ---")
            fine_tuned_model, fine_tuned_metrics = k_fold_cross_validation(
                k=3,
                model_name=model_name,
                dataset=dataset,
                epochs=5,
                fine_tune=True,
                unfreeze_layers=2,  # Fine-tune last 2 layers
                add_extra_conv=False
            )
            results[f"{model_name}_fine_tuned"] = fine_tuned_metrics

            # Save results
            with open('results.pkl', 'wb') as f:
                pickle.dump(results, f)

            # Model with Extra Convolutional Layer
            print("\n--- Model with Extra Convolutional Layer ---")
            extra_conv_model, extra_conv_metrics = k_fold_cross_validation(
                k=3,
                model_name=model_name,
                dataset=dataset,
                epochs=5,
                fine_tune=True,
                unfreeze_layers=2,
                add_extra_conv=True
            )
            results[f"{model_name}_extra_conv"] = extra_conv_metrics

            # Save results
            with open('results.pkl', 'wb') as f:
                pickle.dump(results, f)

        except Exception as e:
            print(f"\nAn error occurred while processing model {model_name}: {e}")
            # Capture the traceback
            error_details = traceback.format_exc()
            print(error_details)
            errors[model_name] = error_details
            # Continue to the next model

        # Optionally, clear CUDA cache to free up memory after each model
        torch.cuda.empty_cache()

    # Print results
    print("\n\n--- Summary of Results ---")
    for key, value in results.items():
        print(f"{key}: Accuracy={value['accuracy']:.4f}, Precision={value['precision']:.4f}, "
              f"Recall={value['recall']:.4f}, F1 Score={value['f1_score']:.4f}")

    # Save final results
    with open('final_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    # Save errors if any
    if errors:
        with open('errors.pkl', 'wb') as f:
            pickle.dump(errors, f)
        print("\nErrors occurred in the following models:")
        for model_name, error_detail in errors.items():
            print(f"{model_name}:\n{error_detail}")

    # Find the best model
    if results:
        best_model_name = max(results, key=lambda k: results[k]['accuracy'])
        print(f"\nBest model is {best_model_name} with accuracy {results[best_model_name]['accuracy']:.4f}")
    else:
        print("No successful results to determine the best model.")

In [15]:
if __name__ == "__main__":
    run_experiments()


Running experiments for model: resnet50

--- Base Model ---

Starting Fold 1


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


Iteration 1 ---------- Best LR = 0.0006430325304733049 ---------- Loss = 0.7091847807168961
Iteration 2 ---------- Best LR = 0.0005787292774259744 ---------- Loss = 0.6934578660875559
Iteration 3 ---------- Best LR = 0.000520856349683377 ---------- Loss = 0.6820209342986345
Iteration 4 ---------- Best LR = 0.0004687707147150393 ---------- Loss = 0.6716282237321138
Iteration 5 ---------- Best LR = 0.0004218936432435354 ---------- Loss = 0.6604954302310944
Epoch [1/5], Loss: 0.6383
Epoch [2/5], Loss: 0.5598
Epoch [3/5], Loss: 0.4634
Epoch [4/5], Loss: 0.3628
Epoch [5/5], Loss: 0.2719
Accuracy: 0.9542
Precision: 0.9466
Recall: 0.9650
F1 Score: 0.9557
Confusion Matrix:
[[231  14]
 [  9 248]]

Starting Fold 2


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Iteration 1 ---------- Best LR = 3.476064767044027e-05 ---------- Loss = 0.6977375708520412
Iteration 2 ---------- Best LR = 3.128458290339624e-05 ---------- Loss = 0.6974752601236105
Iteration 3 ---------- Best LR = 2.8156124613056618e-05 ---------- Loss = 0.6954411715269089
Iteration 4 ---------- Best LR = 2.5340512151750955e-05 ---------- Loss = 0.6950948499143124
Iteration 5 ---------- Best LR = 2.280646093657586e-05 ---------- Loss = 0.6936644054949284
Epoch [1/5], Loss: 0.6917
Epoch [2/5], Loss: 0.6884
Epoch [3/5], Loss: 0.6842
Epoch [4/5], Loss: 0.6783
Epoch [5/5], Loss: 0.6725
Accuracy: 0.5558
Precision: 0.5223
Recall: 0.7395
F1 Score: 0.6122
Confusion Matrix:
[[103 161]
 [ 62 176]]

Starting Fold 3


...

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



An error occurred while processing model densenet121: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.cuda.FloatTensor [32, 1024, 7, 7]], which is output 0 of ReluBackward0, is at version 2; expected version 1 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).
Traceback (most recent call last):
  File "<ipython-input-14-4be056742512>", line 63, in run_experiments
    extra_conv_model, extra_conv_metrics = k_fold_cross_validation(
  File "<ipython-input-13-36e41784db3c>", line 22, in k_fold_cross_validation
    best_lr = simulated_annealing_lr(model, criterion, train_loader)
  File "<ipython-input-11-3db5bcef3cdc>", line 19, in simulated_annealing_lr
    loss.backward()
  File "/usr/local/lib/python3.10/dist-packages/torch/_tensor.py", line 521, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.10/dist-packages/torch/

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Iteration 1 ---------- Best LR = 0.0005565202249604948 ---------- Loss = 0.6146754613146186
Iteration 2 ---------- Best LR = 0.0005008682024644453 ---------- Loss = 0.38972865603864193
Iteration 3 ---------- Best LR = 0.00045078138221800077 ---------- Loss = 0.2801174819469452
Iteration 4 ---------- Best LR = 0.00040570324399620073 ---------- Loss = 0.21716113667935133
Iteration 5 ---------- Best LR = 0.0003651329195965807 ---------- Loss = 0.1830223232973367
Epoch [1/5], Loss: 0.2315
Epoch [2/5], Loss: 0.1360
Epoch [3/5], Loss: 0.1045
Epoch [4/5], Loss: 0.0711
Epoch [5/5], Loss: 0.0680
Accuracy: 0.9681
Precision: 0.9582
Recall: 0.9805
F1 Score: 0.9692
Confusion Matrix:
[[234  11]
 [  5 252]]

Starting Fold 2
Iteration 1 ---------- Best LR = 0.000831110617610465 ---------- Loss = 0.559144395403564
Iteration 2 ---------- Best LR = 0.0007479995558494185 ---------- Loss = 0.29919401044026017
Iteration 3 ---------- Best LR = 0.0006731996002644767 ---------- Loss = 0.18735549482516944
Itera